In [1]:
atomstring = '''
Fe -0.64147387529051 0.51990405379180 0.11450483185168
O 0.33915564253394 2.18819453520299 -0.84159903476570
H 0.65823736209818 2.99799985286513 -0.40575899522591
O -1.29914529040912 -0.20924466129350 -1.80322550106302
H -2.11419692217693 0.03267722517314 -2.27701147100162
O 0.01237770373627 1.24396717111997 2.03533113813669
H 0.83097217272839 1.00756306178418 2.50577204385612
O -1.62284672685044 -1.15048542622041 1.07007991103536
H -1.94592790838151 -1.95766677806096 0.63237073377438
O -2.41866907277644 1.66373032346654 0.25658821238371
H -2.55381601393967 2.56024091790599 -0.09930153080408
O 1.13404022498401 -0.62623224778001 -0.02436092147139
H 1.26370551457750 -1.52400168391753 0.33034524187694
H -3.28110369794222 1.35873477131317 0.59094893393907
H -0.81032015579443 -0.82248643308581 -2.37966496653734
H 1.99795010138314 -0.32692539271158 -0.36001715946551
H 0.57998322602479 2.26642229799559 -1.78148703231078
H -0.47668197075682 1.85673194452426 2.61208351578631
H -1.85653031374812 -1.23356353207299 2.011352050005
'''

In [2]:
import numpy as np
from pyscf import gto, scf, lo, mp, cc

mol = gto.M(atom = atomstring,
            basis = {
                'default': 'sto6g',
                'Fe': 'sto6g'
                },
            verbose = 4,
            unit = 'angstrom',
            symmetry = 0,
            charge = 2,
            spin = 4,
            max_memory = 20000,
            )

mf = scf.UHF(mol).density_fit()
mf = mf.x2c()
mf.chkfile = './hsmf.chk'
mf.init_guess = 'chk'
mf.max_cycle = 100
mf.level_shift = 0.5
mf = mf.newton()
mf.kernel()

stable = False
for i in range(10):
    print(f'mf stability test {i+1}')
    if not stable:
        mo_i, _, stable,_ = mf.stability(return_status=True)
        dm = mf.make_rdm1(mo_i,mf.mo_occ)
        mf = mf.newton()
        mf.kernel(dm0=dm)
    elif stable:
        print(f'mf energy: {mf.e_tot}, stability {stable}')
        break

System: uname_result(system='Linux', node='sharmagroup-rn', release='7.0.0-28-generic', version='#28~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Wed Jul  1 15:50:57 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Thu Aug 13 18:53:05 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 19
[INPUT] num. electrons = 84
[INPUT] charge = 2
[INPUT] spin (= nelec alpha-beta = 2S) = 4
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z  

In [3]:
mymp = mp.MP2(mf).set_frozen()
mymp.kernel()
efull_mp2 = mymp.e_corr
print(f'MP2 Corr = {efull_mp2:.8f}')

mycc = cc.CCSD(mf).set_frozen()
mycc.conv_tol = 1e-6
mycc.conv_tol_normt = 3e-5
mycc.max_cycle = 200
mycc.level_shift = 0.5
mycc.diis_space = 10
mycc.kernel()
efull_ccsd = mycc.e_corr
print(f'CCSD Corr = {efull_ccsd:.8f}')

efull_t = mycc.ccsd_t()
efull_ccsd_t = efull_ccsd + efull_t
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')


******** <class 'pyscf.mp.dfump2.DFUMP2'> ********
nocc = (np.int64(33), np.int64(29)), nmo = (49, 49)
frozen orbitals 11
max_memory 20000 MB (current use 261 MB)
E(DFUMP2) = -1719.53243003494  E_corr = -0.306274942234728
E(SCS-DFUMP2) = -1719.53766217234  E_corr = -0.311507079634863
E_corr(same-spin) = -0.0646417512078582
E_corr(oppo-spin) = -0.24163319102687
MP2 Corr = -0.30627494

******** <class 'pyscf.cc.dfuccsd.UCCSD'> ********
CC2 = 0
CCSD nocc = (np.int64(33), np.int64(29)), nmo = (49, 49)
frozen orbitals 11
max_cycle = 200
direct = 0
conv_tol = 1e-06
conv_tol_normt = 3e-05
diis_space = 10
diis_start_cycle = 0
diis_start_energy_diff = 1e+09
max_memory 20000 MB (current use 265 MB)
Init t2, MP2 energy = -0.306274942234728
Init E_corr(UCCSD) = -0.306274942234728
cycle = 1  E_corr(UCCSD) = -0.346675473236401  dE = -0.040400531  norm(t1,t2) = 0.0675104
cycle = 2  E_corr(UCCSD) = -0.366598059547916  dE = -0.0199225863  norm(t1,t2) = 0.0336451
cycle = 3  E_corr(UCCSD) = -0.387227837

In [4]:
print(f'MP2 Corr = {efull_mp2:.8f}')
print(f'CCSD Corr = {efull_ccsd:.8f}')
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}') 

MP2 Corr = -0.30627494
CCSD Corr = -0.38799823
CCSD(T) Corr = -0.39299942


In [7]:
from pyscf.data import elements
import lno_tools
from pyscf.lno import lnoccsd, ulnoccsd
from pyscf.lno.tools import autofrag_iao

# iao_coeff, iao_frag_list, atm_center = lno_tools.iao_localization(mf)

In [ ]:
moliao = lo.iao.reference_mol(mol)
frag_lolist = autofrag_iao(moliao) # the list of iao assigned to each atom - [[0,1,2],[3,4],[5],...]

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]


In [62]:
def orthonormalize(s1e, mo_coeff, lindep_torr=1e-8):
    # svd combined orbitals
    # s1e = mf.get_ovlp()                # AO overlap, (nao, nao)
    G = mo_coeff.T @ s1e @ mo_coeff      # MO–MO Gram matrix, (n, n), symmetric PSD
    w, V = np.linalg.eigh(G)             # eigh == SVD for symmetric PSD
    w, V = w[::-1], V[:, ::-1]           # descending
    lindep_torr *= w[0]                  # relative threshold
    keep = w > lindep_torr
    rank = keep.sum()
    print(f"union rank = {keep.sum()} of {len(w)}")
    # S-orthonormal orbitals spanning span(moa) ∪ span(mob)
    mo_orth = mo_coeff @ (V[:, keep] / np.sqrt(w[keep]))
    # sanity check: physically orthonormal
    assert np.allclose(mo_orth.T @ s1e @ mo_orth, np.eye(rank), atol=lindep_torr)

    return mo_orth

In [84]:
mol = mf.mol
s1e = mf.get_ovlp()
frozen = elements.chemcore(mol)
moliao = lo.iao.reference_mol(mol)
frag_lolist = autofrag_iao(moliao) # the list of iao assigned to each atom - [[0,1,2],[3,4],[5],...]

orbocc_a = mf.mo_coeff[0][:,frozen:np.count_nonzero(mf.mo_occ[0])]
orbocc_b = mf.mo_coeff[1][:,frozen:np.count_nonzero(mf.mo_occ[1])]
iao_coeff_a = lo.iao.iao(mol, orbocc_a)
iao_coeff_b = lo.iao.iao(mol, orbocc_b)

nfrag = len(frag_lolist)
iao_coeff_c = [None] * nfrag

for ifrag, iao_idx in enumerate(frag_lolist):
    iao_c = np.hstack((iao_coeff_a[:, iao_idx], iao_coeff_b[:, iao_idx]))
    iao_coeff_c[ifrag] = orthonormalize(s1e, iao_c, lindep_torr=1e-5)

iao_coeff_c = np.hstack(iao_coeff_c)
iao_coeff_c = lo.orth.vec_lowdin(iao_coeff_c, s1e)

union rank = 24 of 30
union rank = 8 of 10
union rank = 1 of 2
union rank = 8 of 10
union rank = 2 of 2
union rank = 8 of 10
union rank = 2 of 2
union rank = 8 of 10
union rank = 2 of 2
union rank = 8 of 10
union rank = 2 of 2
union rank = 8 of 10
union rank = 2 of 2
union rank = 2 of 2
union rank = 2 of 2
union rank = 2 of 2
union rank = 1 of 2
union rank = 2 of 2
union rank = 2 of 2


In [82]:
iao_coeff_c = lo.orth.vec_lowdin(iao_coeff_c, s1e)

In [86]:
print(lno_tools.mo_span(iao_coeff_c, s1e, orbocc_a))
print(lno_tools.mo_span(iao_coeff_c, s1e, orbocc_b))
print((iao_coeff_c.T @ s1e @ iao_coeff_c - np.eye(iao_coeff_c.shape[1])).max()) # 2e-6 when lindep = 1e-10

(np.float64(3.2870974830956357e-09), np.float64(0.9989002293136712))
(np.float64(6.206239633321786e-09), np.float64(0.9988943250537765))
0.2776573465506946


In [13]:
nfrozen = elements.chemcore(mol)
nocca = np.count_nonzero(mf.mo_occ[0])
noccb = np.count_nonzero(mf.mo_occ[1])
s1e = mf.get_ovlp()
mo_occa = mf.mo_coeff[0][:,nfrozen:nocca]
mo_occb = mf.mo_coeff[1][:,nfrozen:noccb]
print(lno_tools.mo_span(iao_coeff_a, s1e, mo_occa)) # output 1 = <mo|mo> - <mo|iao><iao|mo>
print(lno_tools.mo_span(iao_coeff_b, s1e, mo_occb)) # output 2 = <iao|iao> - <iao|mo><mo|iao>
print(lno_tools.check_span(mf, (iao_coeff_a, iao_coeff_b), nfrozen)) # for LNO to work, LOs has to span occ MOs

(np.float64(1.239338560704746e-13), np.float64(0.9999901520144607))
(np.float64(2.864040874747072e-13), np.float64(0.9999960429747766))
LO occ span the occupied MO occ space - True.
MO occ span the occupied LO occ space - False.
None


In [6]:
nfrozen = elements.chemcore(mol)
nocca = np.count_nonzero(mf.mo_occ[0])
noccb = np.count_nonzero(mf.mo_occ[1])
s1e = mf.get_ovlp()
mo_occa = mf.mo_coeff[0][:,nfrozen:nocca]
mo_occb = mf.mo_coeff[1][:,nfrozen:noccb]
print(lno_tools.mo_span(iao_coeff[0], s1e, mo_occa)) # output 1 = <mo|mo> - <mo|iao><iao|mo>
print(lno_tools.mo_span(iao_coeff[1], s1e, mo_occb)) # output 2 = <iao|iao> - <iao|mo><mo|iao>
print(lno_tools.check_span(mf, iao_coeff, nfrozen)) # for LNO to work, LOs has to span occ MOs

(np.float64(2.2210849153804502e-13), np.float64(0.9999901520144584))
(np.float64(1.50739154160032e-13), np.float64(0.9999960429747804))
LO occ span the occupied MO occ space - True.
MO occ span the occupied LO occ space - False.
None


In [7]:
lo_coeff = iao_coeff
frag_lolist = iao_frag_list
nfrozen = elements.chemcore(mol)
lno_thresh = 1e-5
run_frag_list = [18] # None - run all fragments, [0, 1, 2, 3, ...] run specific fragments
atom_group = atm_center


print("\n ******* LNO-CALCULATION ******* \n")

lno_tools.check_span(mf, lo_coeff, nfrozen, thresh=1e-10)

if isinstance(mf, scf.rhf.RHF):
    spin_type = "restricted"
    mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=mf.verbose)
elif isinstance(mf, scf.uhf.UHF):
    spin_type = "unrestricted"
    mlno = ulnoccsd.ULNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=mf.verbose)
else:
    raise TypeError(f'unsupported mean-field type: {type(mf)}')

if isinstance(lno_thresh, float):
    mlno.lno_thresh = [lno_thresh*10, lno_thresh]
elif isinstance(lno_thresh, (list, tuple)):
    assert len(lno_thresh) == 2
    mlno.lno_thresh = [lno_thresh[0], lno_thresh[1]]

lno_thresh = mlno.lno_thresh
print(f"LNO THRESHOLD = {mlno.lno_thresh}")
lno_type = ['1h','1h']
eris = mlno.ao2mo()

nfrag_tot = len(frag_lolist)
if run_frag_list is None:
    run_frag_list = range(nfrag_tot)

frag_lolist = [frag_lolist[i] for i in run_frag_list]
nfrag_run = len(frag_lolist)

lno_pct_occ = [None, None]
lno_norb = [[None,None]] * nfrag_tot

las_center = [None]*nfrag_run
las_size = [None]*nfrag_run
lno_emp = np.zeros(nfrag_run, dtype='float64')
lno_ecc  = np.zeros(nfrag_run, dtype='float64')
lno_eqmc = np.zeros(nfrag_run, dtype='float64')

mol = mf.mol

# Loop over fragment
for ifrag, frag_idx in enumerate(run_frag_list):
    
    loidx = frag_lolist[ifrag]

    print("\n")
    width = 80
    msg = f" {spin_type} LNO-FRAGMENT {frag_idx+1}/({nfrag_run},{nfrag_tot}) "
    print(msg.center(width, '='))
    if atom_group is not None:
        loc_ctr = f"{atom_group[frag_idx]}"
        print(f"Center Atom {loc_ctr}")
    else:
        loc_ctr = None

    orbloc, lno_param \
        = lno_tools.get_lnoparam(mlno, lo_coeff, lno_thresh, lno_pct_occ, lno_norb, loidx, ifrag)

    lno_coeff, lno_frozen, uocc_loc, _ \
                = mlno.make_las(eris, orbloc, lno_type, lno_param)
    
    if isinstance(mlno._scf, scf.rhf.RHF):
        lno_frozen, maskact \
            = lnoccsd.get_maskact(lno_frozen, mlno.mo_occ.size)
    elif isinstance(mlno._scf, scf.uhf.UHF):
        lno_frozen, maskact \
            = ulnoccsd.get_maskact(lno_frozen, [mlno.mo_occ[0].size, mlno.mo_occ[1].size])
    else:
        raise TypeError(f'unsupported mean-field type: {type(mlno._scf)}')

    eorb_mp = lno_tools.lnomp2_kernel(mlno, lno_coeff, lno_frozen, uocc_loc, maskact, verbose=4)
    eorb_cc, t1, t2 = \
        lno_tools.lnoccsd_kernel(mlno, lno_coeff, lno_frozen, uocc_loc, maskact, verbose=4)

    print(f'LNO-MP2 Orbital Energy:  {eorb_mp:.8f}')
    print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')

    lno_emp[ifrag] = eorb_mp
    lno_ecc[ifrag] = eorb_cc


 ******* LNO-CALCULATION ******* 

LO occ span the occupied MO occ space - True.
MO occ span the occupied LO occ space - False.
LNO THRESHOLD = [0.0001, 1e-05]


===================== unrestricted LNO-FRAGMENT 19/(1,19) ======================
Center Atom H
LO occ proj: 1 active | 0 standby | 32 orthogonal
LO occ proj: 1 active | 0 standby | 28 orthogonal



WARN: CCSD detected DF being used in the HF object. MO integrals are computed based on the DF 3-index tensors.
It's recommended to use dfccsd.CCSD for the DF-CCSD calculations

Init t2, MP2 energy = -0.045519205598145

WARN: CCSD detected DF being used in the HF object. MO integrals are computed based on the DF 3-index tensors.
It's recommended to use dfccsd.CCSD for the DF-CCSD calculations


******** <class 'pyscf.lno.ulnoccsd.MODIFIED_UCCSD'> ********
CC2 = 0
CCSD nocc = (np.int64(4), np.int64(5)), nmo = (11, 12)
frozen orbitals [array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 51, 52, 53, 54, 55, 56, 57, 58, 59]), array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59])]
max_cycle = 50
direct = 0


In [18]:
elno_mp = sum(lno_emp)
elno_cc = sum(lno_ecc)
print(f"LNO-MP2       = {elno_mp} | CAN-MP2 = {mymp.e_corr}")
print(f"LNO-CCSD      = {elno_cc} | CAN-CCSD = {mycc.e_corr}")
print(f"LNO-CCSD+DMP2 = {elno_cc + mymp.e_corr - elno_mp}")

LNO-MP2       = -0.2999295133940991 | CAN-MP2 = -0.30627494222304563
LNO-CCSD      = -0.38147891054325456 | CAN-CCSD = -0.3879982328937469
LNO-CCSD+DMP2 = -0.38782433937220107


In [ ]:
# split lno_coeff into frzocc actocc actvir frzvir
lno_split = lno_tools.split_lno(mlno, lno_coeff, lno_frozen)
lno_acta = np.hstack((lno_split[0][1], lno_split[0][2]))
lno_actb = np.hstack((lno_split[1][1], lno_split[1][2]))

LAS info
nfrozen occupied orbitals:  [40, 35]
nactive occupied orbitals:  [4, 5]
nactive virtual orbitals:   [7, 7]
nfrozen virtual orbitals:   [9, 13]


In [10]:
lno_tools.check_rspan(lno_acta, s1e, lno_actb)

(np.float64(0.7982506350915672), np.float64(0.10533187880536266))